# 06 — Prediction Benchmark: Hyperparameter-Tuned RF, GPR, and SVR

**Wavelength-Resolved Photo-Impedance and Leakage-Aware Machine Learning Reveal a
Schottky-to-Plasmonic Mechanism Transition in Ag/Au/g-C3N4 Photocathodes for
Hydrogen Evolution**

This notebook runs a systematic hyperparameter search for Random Forest, Gaussian
Process Regression, and Support Vector Regression on all five targets, using the
same grouped leave-one-out cross-validation and target-specific feature exclusions
as every other model notebook in this project (`utils.py`). It asks a single
question: does hyperparameter tuning of these three model classes beat the models
already reported in `05_GPR_models.ipynb` (LASSO for photocurrent, GPR for the
three EIS-derived targets)?

**Input:** `Results/MASTER_TABLE_n84.csv`
**Run after** `01_data_preparation.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
from sklearn.svm import SVR
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore")

from utils import load_master, prepare, grouped_loo_predict, bootstrap_r2_ci, TARGETS

df = load_master()
print(f"Master table: {df.shape}")
print(f"Unique (sample, condition) groups: {df['group_id'].nunique()}")

Master table: (84, 49)
Unique (sample, condition) groups: 28


## 1. Random Forest — Hyperparameter Search

Three hyperparameters are varied: `max_features` (fraction of features considered
per split), `min_samples_leaf` (minimum leaf size), and `max_depth` (tree depth
cap). Every configuration is scored by grouped LOOCV directly — there is no
row-wise pass, since the catalyst x condition grouping is required for a valid
estimate on this dataset (see `utils.py`).

In [2]:
rf_param_grid = {
    "max_features": ["sqrt", 0.5, 1.0],
    "min_samples_leaf": [2, 4],
    "max_depth": [None, 5, 10],
}
N_ESTIMATORS_RF = 500

rf_results = []
for target_col, target_name in TARGETS.items():
    X, y, groups, cols = prepare(df, target_col)
    for params in ParameterGrid(rf_param_grid):
        factory = lambda p=params: RandomForestRegressor(
            **p, n_estimators=N_ESTIMATORS_RF, random_state=42, n_jobs=-1)
        y_loo = grouped_loo_predict(factory, X, y, groups, scale=False)
        rf_results.append({"Target": target_name, **params,
                            "R2_GroupedLOOCV": round(r2_score(y, y_loo), 4)})

df_rf = pd.DataFrame(rf_results)
best_rf = (df_rf.sort_values("R2_GroupedLOOCV", ascending=False)
                 .groupby("Target", as_index=False).first())

print(f"Grid size: {len(ParameterGrid(rf_param_grid))} configurations x 5 targets "
      f"= {len(df_rf)} fits")
print("\nBest RF configuration per target (grouped LOOCV):")
print(best_rf[["Target", "max_features", "min_samples_leaf", "max_depth",
                "R2_GroupedLOOCV"]].to_string(index=False))
df_rf.to_csv("Results/benchmark_rf_grid.csv", index=False)

Grid size: 18 configurations x 5 targets = 90 fits

Best RF configuration per target (grouped LOOCV):
                    Target  max_features  min_samples_leaf  max_depth  R2_GroupedLOOCV
Charge transfer resistance           0.5                 2       10.0           0.9128
      LSPR resistance drop           1.0                 2       10.0           0.8233
   Photocurrent J (-0.7 V)           1.0                 4       10.0           0.4974
           Relaxation time           0.5                 2       10.0           0.8043
               Tafel slope           1.0                 4        5.0          -0.3076


## 2. Gaussian Process Regression — Kernel Search

Three kernel families are compared, each paired with a `WhiteKernel` noise term,
consistent with the kernel structure used for the reported GPR models in
Notebook 05.

In [3]:
gpr_kernels = {
    "RBF":        ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))
                  + WhiteKernel(1e-2, (1e-6, 1e1)),
    "Matern_1.5": ConstantKernel(1.0, (1e-3, 1e3)) * Matern(1.0, (1e-2, 1e2), nu=1.5)
                  + WhiteKernel(1e-2, (1e-6, 1e1)),
    "Matern_2.5": ConstantKernel(1.0, (1e-3, 1e3)) * Matern(1.0, (1e-2, 1e2), nu=2.5)
                  + WhiteKernel(1e-2, (1e-6, 1e1)),
}

gpr_results = []
for target_col, target_name in TARGETS.items():
    X, y, groups, cols = prepare(df, target_col)
    for kname, kernel in gpr_kernels.items():
        factory = lambda k=kernel: GaussianProcessRegressor(
            kernel=k, alpha=1e-8, normalize_y=True, random_state=42, n_restarts_optimizer=3)
        y_loo = grouped_loo_predict(factory, X, y, groups, scale=True)
        gpr_results.append({"Target": target_name, "Kernel": kname,
                             "R2_GroupedLOOCV": round(r2_score(y, y_loo), 4)})

df_gpr = pd.DataFrame(gpr_results)
best_gpr = (df_gpr.sort_values("R2_GroupedLOOCV", ascending=False)
                   .groupby("Target", as_index=False).first())

print("Best GPR kernel per target (grouped LOOCV):")
print(best_gpr.to_string(index=False))
df_gpr.to_csv("Results/benchmark_gpr_grid.csv", index=False)

Best GPR kernel per target (grouped LOOCV):
                    Target     Kernel  R2_GroupedLOOCV
Charge transfer resistance Matern_1.5           0.9297
      LSPR resistance drop        RBF           0.9070
   Photocurrent J (-0.7 V) Matern_1.5           0.2383
           Relaxation time Matern_2.5           0.9048
               Tafel slope Matern_1.5          -1.4497


## 3. Support Vector Regression — Hyperparameter Search

`kernel`, `gamma`, `C` (regularization), and `epsilon` (tolerance band) are varied
over a grid of 120 configurations per target.

In [4]:
svr_param_grid = {
    "kernel":  ["rbf", "poly"],
    "gamma":   ["scale", 0.01, 0.1, 1.0],
    "C":       [1.0, 10.0, 100.0, 1000.0, 10000.0],
    "epsilon": [0.01, 0.1, 0.5],
}

svr_results = []
for target_col, target_name in TARGETS.items():
    X, y, groups, cols = prepare(df, target_col)
    for params in ParameterGrid(svr_param_grid):
        factory = lambda p=params: SVR(**p)
        y_loo = grouped_loo_predict(factory, X, y, groups, scale=True)
        svr_results.append({"Target": target_name, **params,
                             "R2_GroupedLOOCV": round(r2_score(y, y_loo), 4)})

df_svr = pd.DataFrame(svr_results)
best_svr = (df_svr.sort_values("R2_GroupedLOOCV", ascending=False)
                   .groupby("Target", as_index=False).first())

print(f"Grid size: {len(ParameterGrid(svr_param_grid))} configurations x 5 targets "
      f"= {len(df_svr)} fits")
print("\nBest SVR configuration per target (grouped LOOCV):")
print(best_svr[["Target", "kernel", "C", "epsilon", "gamma",
                 "R2_GroupedLOOCV"]].to_string(index=False))
df_svr.to_csv("Results/benchmark_svr_grid.csv", index=False)

Grid size: 120 configurations x 5 targets = 600 fits

Best SVR configuration per target (grouped LOOCV):
                    Target kernel       C  epsilon  gamma  R2_GroupedLOOCV
Charge transfer resistance    rbf 10000.0     0.50   0.01           0.8984
      LSPR resistance drop    rbf 10000.0     0.10   0.01           0.8370
   Photocurrent J (-0.7 V)    rbf     1.0     0.01   0.01           0.4488
           Relaxation time    rbf  1000.0     0.01   0.01           0.8635
               Tafel slope   poly     1.0     0.10   0.01          -0.0071


## 4. Best Tuned Model per Target

In [5]:
summary_rows = []
for target_name in TARGETS.values():
    candidates = [
        ("RF",  best_rf.loc[best_rf.Target == target_name, "R2_GroupedLOOCV"].values[0]),
        ("GPR", best_gpr.loc[best_gpr.Target == target_name, "R2_GroupedLOOCV"].values[0]),
        ("SVR", best_svr.loc[best_svr.Target == target_name, "R2_GroupedLOOCV"].values[0]),
    ]
    best_model, best_r2 = max(candidates, key=lambda c: c[1])
    summary_rows.append({"Target": target_name, "Best tuned model": best_model,
                          "R2_GroupedLOOCV": round(best_r2, 4)})

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("Results/benchmark_best_tuned_per_target.csv", index=False)

                    Target Best tuned model  R2_GroupedLOOCV
   Photocurrent J (-0.7 V)               RF           0.4974
Charge transfer resistance              GPR           0.9297
      LSPR resistance drop              GPR           0.9070
           Relaxation time              GPR           0.9048
               Tafel slope              SVR          -0.0071


## 5. Bootstrap 95% Confidence Intervals — Best Tuned Model per Target

Same resampling scheme as every other CI in this project: 2000 resamples of the
28 catalyst x condition groups (not individual rows).

In [6]:
model_factories = {
    "RF":  lambda p: RandomForestRegressor(
               max_features=p["max_features"], min_samples_leaf=int(p["min_samples_leaf"]),
               max_depth=(None if pd.isna(p["max_depth"]) else int(p["max_depth"])),
               n_estimators=N_ESTIMATORS_RF, random_state=42, n_jobs=-1),
    "SVR": lambda p: SVR(kernel=p["kernel"], C=float(p["C"]), epsilon=float(p["epsilon"]),
                          gamma=p["gamma"]),
    "GPR": lambda p: GaussianProcessRegressor(
               kernel=gpr_kernels[p["Kernel"]], alpha=1e-8, normalize_y=True,
               random_state=42, n_restarts_optimizer=3),
}
scale_by_model = {"RF": False, "SVR": True, "GPR": True}
best_table_by_model = {"RF": best_rf, "SVR": best_svr, "GPR": best_gpr}

ci_results = []
print("="*90)
print(f"{'Target':<28} {'Model':<6} {'R2_point':>10} {'CI_low':>8} {'CI_high':>8} {'CI_width':>9}")
print("="*90)

for target_col, target_name in TARGETS.items():
    row = df_summary.loc[df_summary.Target == target_name].iloc[0]
    model_name = row["Best tuned model"]
    params_row = best_table_by_model[model_name].loc[
        best_table_by_model[model_name].Target == target_name].iloc[0]

    X, y, groups, cols = prepare(df, target_col)
    factory = lambda p=params_row, m=model_name: model_factories[m](p)
    y_loo = grouped_loo_predict(factory, X, y, groups, scale=scale_by_model[model_name])
    r2_point = r2_score(y, y_loo)
    ci_lo, ci_hi = bootstrap_r2_ci(y, y_loo, groups)

    print(f"{target_name:<28} {model_name:<6} {r2_point:>10.4f} "
          f"{ci_lo:>8.3f} {ci_hi:>8.3f} {ci_hi - ci_lo:>9.3f}")

    ci_results.append({"Target": target_name, "Model": model_name,
                        "R2_point": round(r2_point, 4),
                        "CI_low": round(ci_lo, 4), "CI_high": round(ci_hi, 4)})

print("="*90)
df_ci = pd.DataFrame(ci_results)
df_ci.to_csv("Results/benchmark_bootstrap_ci.csv", index=False)

Target                       Model    R2_point   CI_low  CI_high  CI_width
Photocurrent J (-0.7 V)      RF         0.4974    0.196    0.707     0.511
Charge transfer resistance   GPR        0.9297    0.902    0.959     0.056
LSPR resistance drop         GPR        0.9070    0.847    0.938     0.092
Relaxation time              GPR        0.9048    0.870    0.943     0.073
Tafel slope                  SVR       -0.0071   -0.161   -0.006     0.155


## 6. Comparison Against the Reported Models (Notebook 05)

The reported model per target — LASSO (alpha=0.05) for photocurrent, GPR for the
three EIS-derived targets — comes from `05_GPR_models.ipynb`, with its own
grouped-LOOCV point estimate and bootstrap CI. A tuned model here is only a
genuine improvement if its CI clears the reported model's CI rather than merely
overlapping it.

In [7]:
reported_models = {
    "Photocurrent J (-0.7 V)":     {"Model": "LASSO (alpha=0.05)", "R2_point": 0.5327,
                                      "CI_low": 0.269, "CI_high": 0.766},
    "Charge transfer resistance":  {"Model": "GPR", "R2_point": 0.9088,
                                      "CI_low": 0.877, "CI_high": 0.947},
    "LSPR resistance drop":        {"Model": "LASSO (alpha=0.01)", "R2_point": 0.8126,
                                      "CI_low": 0.658, "CI_high": 0.895},
    "Relaxation time":             {"Model": "GPR", "R2_point": 0.8602,
                                      "CI_low": 0.827, "CI_high": 0.898},
    "Tafel slope":                 {"Model": "RF", "R2_point": -0.5517,
                                      "CI_low": -1.003, "CI_high": -0.268},
}

print("="*105)
print(f"{'Target':<28} {'Reported (05)':<22} {'Tuned (best, 06)':<22} {'Verdict':<35}")
print("="*105)

for target_name in TARGETS.values():
    rep = reported_models[target_name]
    tuned = df_ci.loc[df_ci.Target == target_name].iloc[0]

    overlap = not (tuned["CI_high"] < rep["CI_low"] or tuned["CI_low"] > rep["CI_high"])
    clears = tuned["CI_low"] > rep["CI_high"]
    verdict = ("tuned model clears reported CI" if clears else
               "CIs overlap - no defensible improvement" if overlap else
               "CIs do not overlap, tuned is lower")

    print(f"{target_name:<28} "
          f"{rep['Model']+' '+format(rep['R2_point'],'.3f'):<22} "
          f"{tuned['Model']+' '+format(tuned['R2_point'],'.3f'):<22} "
          f"{verdict:<35}")

Target                       Reported (05)          Tuned (best, 06)       Verdict                            
Photocurrent J (-0.7 V)      LASSO (alpha=0.05) 0.533 RF 0.497               CIs overlap - no defensible improvement
Charge transfer resistance   GPR 0.909              GPR 0.930              CIs overlap - no defensible improvement
LSPR resistance drop         LASSO (alpha=0.01) 0.813 GPR 0.907              CIs overlap - no defensible improvement
Relaxation time              GPR 0.860              GPR 0.905              CIs overlap - no defensible improvement
Tafel slope                  RF -0.552              SVR -0.007             tuned model clears reported CI     


No target shows a statistically defensible improvement over the models already
reported in Notebook 05: every tuned RF/GPR/SVR candidate's confidence interval
overlaps the corresponding reported model's interval, so none clears it outright.
LASSO remains the reported model for photocurrent and GPR remains the reported
model for the three EIS-derived targets.

Tafel slope is the one partial exception: the best tuned model (SVR) has a less
negative point estimate than the reported RF baseline, and their intervals do not
overlap. This is a genuine, if modest, result — but since the tuned SVR's R² is
still at or below zero, it does not make Tafel slope a usable target; the
descriptor set in this project simply does not carry predictive signal for it.

**Bottom line:** systematic hyperparameter tuning of RF, GPR, and SVR across three
model classes and hundreds of configurations does not change the model choices
already reported in Notebook 05. This benchmark is reported as a robustness check
on those choices, not as a source of new results.